In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from acc import get_f1_score, get_user_accuracy, get_producer_accuracy, get_total_accuracy

In [3]:
agg_dir_all = Path('aggregated_validation_dist_2024')
agg_dirs = sorted(list(agg_dir_all.glob('*/')))
sample_agg_dir = agg_dirs[0]
sample_agg_dir

PosixPath('aggregated_validation_dist_2024/builtnewalert__14RNU')

In [4]:
def get_paths(agg_dir: Path):
    veg_status_path = list(agg_dir.glob('dist_hls_veg_*.tif'))[0]
    gen_status_path = list(agg_dir.glob('dist_hls_gen_*.tif'))[0]
    dist_s1_path = list(agg_dir.glob('dist_s1_agg*.tif'))[0]
    dist_s1_count = list(agg_dir.glob('*_count_2024_s1.tif'))[0]
    dist_hls_count = list(agg_dir.glob('*_count_2024_hls.tif'))[0]
    lc_path = list(agg_dir.glob('*glad_lc_mask.tif'))[0]
    return {'veg_status': veg_status_path,
            'gen_status': gen_status_path,
            'dist_s1': dist_s1_path,
            'dist_s1_count': dist_s1_count,
            'dist_hls_count': dist_hls_count,
            'lc': lc_path}


In [5]:
def open_one(path: Path) -> np.ndarray: 
    with rasterio.open(path) as src:
        return src.read(1)

def compute_accuracy(ref_arr, pred_arr, valid_mask):
    ref_arr = ref_arr[valid_mask]
    pred_arr = pred_arr[valid_mask]

    f1 = get_f1_score(ref_arr, pred_arr)
    user_acc = get_user_accuracy(ref_arr, pred_arr)
    producer_acc = get_producer_accuracy(ref_arr, pred_arr)
    total_acc = get_total_accuracy(ref_arr, pred_arr)
    count = np.sum(valid_mask)

    return {'f1': float(f1), 
            'user_acc': float(user_acc), 
            'producer_acc': float(producer_acc), 
            'total_acc': float(total_acc), 
            'count': int(count)}

    

def analysis_for_agg_dir(agg_dir: Path, min_s1_obs: int = 10, min_hls_obs: int = 20):
    
    out_data = {}

    path_dict = get_paths(agg_dir)
    veg_status = open_one(path_dict['veg_status'])
    gen_status = open_one(path_dict['gen_status'])
    dist_s1 = open_one(path_dict['dist_s1'])
    dist_s1_count = open_one(path_dict['dist_s1_count'])
    dist_hls_count = open_one(path_dict['dist_hls_count'])
    lc = open_one(path_dict['lc'])

    s1_mask_count = dist_s1_count > min_s1_obs
    hls_mask_count = dist_hls_count > min_hls_obs
    veg_confirmed_all = (np.isin(veg_status, [3, 6])).astype(np.uint8)
    #veg_confirmed_high = (np.isin(veg_status, [6])).astype(np.uint8)
    gen_confirmed_all = (np.isin(gen_status, [3, 6])).astype(np.uint8)
    # gen_confirmed_high = (np.isin(gen_status, [6])).astype(np.uint8)
    dist_s1_confirmed_all = (np.isin(dist_s1, [3, 6])).astype(np.uint8)

    mask_count = s1_mask_count & hls_mask_count

    mask_dict = {'desert': np.isin(lc, [0, 1]),
                 'semi_arid': np.isin(lc, list(range(2, 19))),
                 'dense_short_vegetation': np.isin(lc, list(range(20, 25))),
                 'tree_cover_all': np.isin(lc, list(range(25, 39))),
                 'tree_cover_above_10m': np.isin(lc, list(range(32, 39))),
                 'wetland_tree_cover': np.isin(lc, list(range(125, 149))),
                 'wetland_vegetation_sparse': np.isin(lc, list(range(102, 119))),
                 'wetland_other': np.isin(lc, list(range(119, 125)) + [100, 101]),
                 'surface_water': np.isin(lc, list(range(200, 209))),
                 'built_up': lc == 250,
                 'cropland': lc == 244,
                 'ocean': lc == 254,
                 'snow_ice': lc == 241,
                 }

    out_data['accuracy_gen_all_confirmed'] = compute_accuracy(gen_confirmed_all, dist_s1_confirmed_all, mask_count)
    # out_data['accuracy_gen_high_confirmed'] = compute_accuracy(veg_confirmed_high, dist_s1_confirmed_all, mask_count)
    out_data['accuracy_veg_all_confirmed'] = compute_accuracy(veg_confirmed_all, dist_s1_confirmed_all, mask_count)
    # out_data['accuracy_veg_high_confirmed'] = compute_accuracy(veg_confirmed_high, dist_s1_confirmed_all, mask_count)
    for key, mask in mask_dict.items():
        mask_lc_valid = mask & mask_count
        out_data[f'accuracy_gen_all_confirmed_{key}'] = compute_accuracy(gen_confirmed_all, dist_s1_confirmed_all, mask_lc_valid)
        out_data[f'accuracy_veg_all_confirmed_{key}'] = compute_accuracy(veg_confirmed_all, dist_s1_confirmed_all, mask_lc_valid)
        # out_data[f'accuracy_gen_high_confirmed_{key}'] = compute_accuracy(gen_confirmed_high, dist_s1_confirmed_all, mask_lc_valid)
        # out_data[f'accuracy_veg_high_confirmed_{key}'] = compute_accuracy(veg_confirmed_high, dist_s1_confirmed_all, mask_lc_valid)
    out_data['umd_name'] = agg_dir.stem
    return out_data
    

In [6]:
sample_agg_dir = agg_dirs[70]
print(sample_agg_dir)
analysis_for_agg_dir(sample_agg_dir)

aggregated_validation_dist_2024/treelosswet__49RGN


/Users/cmarshak/bekaert-team/dist-s1-research/marshak/Zg_validation_variation/acc.py:14: RuntimeWarning: invalid value encountered in scalar divide
  f1 = 2 * tp / (2 * tp + fp + fn)
/Users/cmarshak/bekaert-team/dist-s1-research/marshak/Zg_validation_variation/acc.py:20: RuntimeWarning: invalid value encountered in scalar divide
  user_accuracy = tp / (tp + fp)
/Users/cmarshak/bekaert-team/dist-s1-research/marshak/Zg_validation_variation/acc.py:26: RuntimeWarning: invalid value encountered in scalar divide
  producer_accuracy = tp / (tp + fn)
/Users/cmarshak/bekaert-team/dist-s1-research/marshak/Zg_validation_variation/acc.py:32: RuntimeWarning: invalid value encountered in scalar divide
  total_accuracy = (tp + tn) / (tp + tn + fp + fn)


{'accuracy_gen_all_confirmed': {'f1': 0.23539831817372797,
  'user_acc': 0.17237174838755107,
  'producer_acc': 0.3710819278005606,
  'total_acc': 0.9309841016427683,
  'count': 13133684},
 'accuracy_veg_all_confirmed': {'f1': 0.23410780745655893,
  'user_acc': 0.22834204053698473,
  'producer_acc': 0.24017229505853613,
  'total_acc': 0.9079150221674285,
  'count': 13133684},
 'accuracy_gen_all_confirmed_desert': {'f1': 0.0,
  'user_acc': 0.0,
  'producer_acc': 0.0,
  'total_acc': 0.8787878787878788,
  'count': 33},
 'accuracy_veg_all_confirmed_desert': {'f1': 0.0,
  'user_acc': 0.0,
  'producer_acc': 0.0,
  'total_acc': 0.9393939393939394,
  'count': 33},
 'accuracy_gen_all_confirmed_semi_arid': {'f1': 0.21245421245421245,
  'user_acc': 0.14720812182741116,
  'producer_acc': 0.3815789473684211,
  'total_acc': 0.895224171539961,
  'count': 2052},
 'accuracy_veg_all_confirmed_semi_arid': {'f1': 0.05172413793103448,
  'user_acc': 0.030456852791878174,
  'producer_acc': 0.1714285714285714

In [7]:
all_data = [analysis_for_agg_dir(agg_dir) for agg_dir in tqdm(agg_dirs)]

100%|██████████| 85/85 [01:24<00:00,  1.01it/s]


In [8]:
import json

with open('acc.json', 'w') as f:
    json.dump(all_data, f, indent=2)

# Aggregations

## Agg by Type

In [9]:
types = [agg_dir.stem.split('__')[0] for agg_dir in agg_dirs]
types = list(set(types))
types

['waternew',
 'gen',
 'builtnewalert',
 'fire',
 'treelosswet',
 'cropnew',
 'oldcrop_short',
 'other',
 'treelossTF',
 'wetshort']

In [10]:
df_acc = pd.json_normalize(all_data)
df_acc.head()

,umd_name,accuracy_gen_all_confirmed.f1,accuracy_gen_all_confirmed.user_acc,accuracy_gen_all_confirmed.producer_acc,accuracy_gen_all_confirmed.total_acc,accuracy_gen_all_confirmed.count,accuracy_veg_all_confirmed.f1,accuracy_veg_all_confirmed.user_acc,accuracy_veg_all_confirmed.producer_acc,accuracy_veg_all_confirmed.total_acc,...,accuracy_gen_all_confirmed_snow_ice.f1,accuracy_gen_all_confirmed_snow_ice.user_acc,accuracy_gen_all_confirmed_snow_ice.producer_acc,accuracy_gen_all_confirmed_snow_ice.total_acc,accuracy_gen_all_confirmed_snow_ice.count,accuracy_veg_all_confirmed_snow_ice.f1,accuracy_veg_all_confirmed_snow_ice.user_acc,accuracy_veg_all_confirmed_snow_ice.producer_acc,accuracy_veg_all_confirmed_snow_ice.total_acc,accuracy_veg_all_confirmed_snow_ice.count
0,builtnewalert__14RNU,0.178646,0.308600,0.125709,0.958266,13392161,0.120196,0.259173,0.078241,0.944200,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
1,builtnewalert__16SBF,0.201754,0.133620,0.411669,0.823637,13392666,0.340996,0.302235,0.391161,0.805148,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
2,builtnewalert__22KFA,0.668697,0.593488,0.765734,0.897117,231544,0.621869,0.690530,0.565628,0.853086,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
3,builtnewalert__22LDQ,0.433678,0.380843,0.503533,0.882231,12719570,0.405349,0.575186,0.312945,0.800185,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
4,builtnewalert__22MDT,0.194235,0.172788,0.221760,0.926765,11091406,0.242595,0.496196,0.160543,0.841721,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0


In [11]:
df_acc['umd_type'] = df_acc.umd_name.map(lambda x: x.split('__')[0])

In [12]:
df_by_type = df_acc[[col for col in df_acc.columns if col != 'umd_name']].groupby('umd_type').mean()

In [13]:
df_by_type.to_csv('acc_by_type.csv')

In [14]:
df_acc.to_csv('acc_by_tile.csv')

# Agg by Mask

In [15]:
import pandas as pd

acc_cols = [c for c in df_acc.select_dtypes(include=['float']) if 'count' not in c]
count_cols = [c for c in df_acc.columns if 'count' in c]
groups = list(set([c.split('.')[0] for c in acc_cols]))
acc_types = list(set([c.split('.')[1] for c in acc_cols]))
acc_types

['f1', 'user_acc', 'total_acc', 'producer_acc']

In [16]:
df_acc.head()

,umd_name,accuracy_gen_all_confirmed.f1,accuracy_gen_all_confirmed.user_acc,accuracy_gen_all_confirmed.producer_acc,accuracy_gen_all_confirmed.total_acc,accuracy_gen_all_confirmed.count,accuracy_veg_all_confirmed.f1,accuracy_veg_all_confirmed.user_acc,accuracy_veg_all_confirmed.producer_acc,accuracy_veg_all_confirmed.total_acc,...,accuracy_gen_all_confirmed_snow_ice.user_acc,accuracy_gen_all_confirmed_snow_ice.producer_acc,accuracy_gen_all_confirmed_snow_ice.total_acc,accuracy_gen_all_confirmed_snow_ice.count,accuracy_veg_all_confirmed_snow_ice.f1,accuracy_veg_all_confirmed_snow_ice.user_acc,accuracy_veg_all_confirmed_snow_ice.producer_acc,accuracy_veg_all_confirmed_snow_ice.total_acc,accuracy_veg_all_confirmed_snow_ice.count,umd_type
0,builtnewalert__14RNU,0.178646,0.308600,0.125709,0.958266,13392161,0.120196,0.259173,0.078241,0.944200,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,builtnewalert
1,builtnewalert__16SBF,0.201754,0.133620,0.411669,0.823637,13392666,0.340996,0.302235,0.391161,0.805148,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,builtnewalert
2,builtnewalert__22KFA,0.668697,0.593488,0.765734,0.897117,231544,0.621869,0.690530,0.565628,0.853086,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,builtnewalert
3,builtnewalert__22LDQ,0.433678,0.380843,0.503533,0.882231,12719570,0.405349,0.575186,0.312945,0.800185,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,builtnewalert
4,builtnewalert__22MDT,0.194235,0.172788,0.221760,0.926765,11091406,0.242595,0.496196,0.160543,0.841721,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,builtnewalert


In [17]:
def lc_extractor(group: str):
    if '.' in group:
        group = group.split('.')[0]
    if 'confirmed_' in group:
        return group.split('confirmed_')[1]
    else:
        return 'all_classes'
    
def status_extractor(group: str):
    if '_gen_' in group:
        return 'gen'
    elif '_veg_' in group:
        return 'veg'
    else:
        raise ValueError(f'Unknown group: {group}')

In [18]:
def format_acc(df: pd.DataFrame, acc_col: str):

    acc_type = acc_col.split('.')[1]
    df_expanded_group = df[['umd_name', acc_col]].reset_index(drop=True)
    df_expanded_group = df_expanded_group.rename(columns={acc_col: 'acc'})
    df_expanded_group['lc_class'] = lc_extractor(acc_col)
    df_expanded_group['hls_type'] = status_extractor(acc_col)
    df_expanded_group['acc_type'] = acc_type
    return df_expanded_group

dfs = [format_acc(df_acc, col) for col in df_acc.columns if pd.api.types.is_numeric_dtype(df_acc[col]) and 'count' not in col]

In [19]:
df_acc_expanded = pd.concat(dfs, axis=0)
df_acc_expanded.sort_values(by='acc', ascending=False, inplace=True)
df_acc_expanded.head()

,umd_name,acc,lc_class,hls_type,acc_type
33,gen__47TNH,1.0,surface_water,gen,producer_acc
23,fire__22LDR,1.0,wetland_vegetation_sparse,gen,total_acc
60,treelossTF__35VLF,1.0,desert,veg,total_acc
59,treelossTF__33NUF,1.0,wetland_vegetation_sparse,gen,total_acc
33,gen__47TNH,1.0,dense_short_vegetation,gen,total_acc


In [20]:
def get_acc_by_group(group: str, 
                     acc_type: str,
                     ref_acc_column_key: str = 'f1',
                     top_percent: float = 0.8):
    count_col = group + '.count'
    acc_col = group + '.' + acc_type
    if acc_type != ref_acc_column_key:
        ref_acc_col = group + '.' + ref_acc_column_key
        cols = [count_col, acc_col, ref_acc_col]
    else:
        ref_acc_col = acc_col
        cols = [count_col, acc_col]
    df_acc_temp = df_acc[cols].sort_values(by=ref_acc_col, ascending=False)
    n = df_acc_temp.shape[0]
    df_acc_temp = df_acc_temp.iloc[:int(n * top_percent)]
    total_count = df_acc_temp[count_col].sum()
    weighted_acc = (df_acc_temp[acc_col] * df_acc_temp[count_col]).sum() / total_count
    return float(weighted_acc)

In [21]:
agg_by_tile = [{'group': g, 'acc_type': a, 'acc': get_acc_by_group(g, a)}  for a in acc_types for g in groups]
df_agg_by_tile = pd.DataFrame(agg_by_tile)

df_agg_by_tile_f = df_agg_by_tile.copy()
df_agg_by_tile_f['lc_class'] = df_agg_by_tile_f['group'].map(lc_extractor)
df_agg_by_tile_f['hls_type'] = df_agg_by_tile_f['group'].map(status_extractor)
df_agg_by_tile_f.drop(columns=['group'], inplace=True)
df_agg_by_tile_f.head()

,acc_type,acc,lc_class,hls_type
0,f1,0.333655,tree_cover_all,veg
1,f1,0.008431,desert,veg
2,f1,0.360249,wetland_other,veg
3,f1,0.290536,dense_short_vegetation,veg
4,f1,0.046578,ocean,veg


In [22]:
df_agg_by_tile_f = df_agg_by_tile_f.sort_values(by=['acc_type','acc'], ascending=False)
df_agg_by_tile_f = df_agg_by_tile_f[['acc_type', 'lc_class', 'hls_type', 'acc']]
df_agg_by_tile_f.to_csv('acc_by_tile_group_best.csv', index=False)



In [23]:
dfs = []
for acc_type in acc_types:
    df_agg_by_tile_temp = df_agg_by_tile_f[df_agg_by_tile_f['acc_type'] == acc_type].reset_index(drop=True)
    df_agg_by_tile_temp.drop(columns=['acc_type'], inplace=True)
    df_agg_by_tile_temp.rename(columns={'acc': acc_type}, inplace=True)
    dfs.append(df_agg_by_tile_temp)

df_all = pd.DataFrame(columns=['lc_class', 'hls_type'])
for df in dfs[:]:
    df_all = pd.merge(df_all, df, on=['lc_class', 'hls_type'], how='right') 
df_all.sort_values(by=['user_acc'], ascending=False, inplace=True)
df_all.head()


,lc_class,hls_type,f1,user_acc,total_acc,producer_acc
23,wetland_tree_cover,veg,0.348435,0.689469,0.814895,0.277308
18,tree_cover_above_10m,veg,0.366355,0.557727,0.901500,0.340346
21,tree_cover_all,veg,0.333655,0.530702,0.884830,0.300911
19,wetland_tree_cover,gen,0.324895,0.488417,0.846320,0.335133
9,tree_cover_above_10m,gen,0.356688,0.397613,0.903188,0.432946


In [24]:
df_all_user = df_all.sort_values(by=['user_acc'], ascending=False)
df_all_user = df_all_user[df_all_user['producer_acc'] > .2].reset_index(drop=True)
df_all_user.head()

,lc_class,hls_type,f1,user_acc,total_acc,producer_acc
0,wetland_tree_cover,veg,0.348435,0.689469,0.814895,0.277308
1,tree_cover_above_10m,veg,0.366355,0.557727,0.901500,0.340346
2,tree_cover_all,veg,0.333655,0.530702,0.884830,0.300911
3,wetland_tree_cover,gen,0.324895,0.488417,0.846320,0.335133
4,tree_cover_above_10m,gen,0.356688,0.397613,0.903188,0.432946


In [25]:
df_all_prod = df_all.sort_values(by=['producer_acc'], ascending=False)
df_all_prod = df_all_prod[df_all_prod['user_acc'] > .2].reset_index(drop=True)
df_all_prod.head()

,lc_class,hls_type,f1,user_acc,total_acc,producer_acc
0,wetland_other,gen,0.315785,0.228900,0.666476,0.611289
1,wetland_other,veg,0.360249,0.306287,0.663326,0.571381
2,surface_water,gen,0.294379,0.230759,0.832900,0.537016
3,all_classes,gen,0.287680,0.239310,0.847129,0.445748
4,tree_cover_above_10m,gen,0.356688,0.397613,0.903188,0.432946


In [26]:
df_all_no_lc = df_all[df_all['lc_class'] == 'all_classes'].sort_values(by=['user_acc'], ascending=False)
df_all_no_lc.head()

,lc_class,hls_type,f1,user_acc,total_acc,producer_acc
14,all_classes,veg,0.324252,0.352195,0.823221,0.388081
6,all_classes,gen,0.287680,0.239310,0.847129,0.445748


In [27]:
from excel_export import df_to_excel_with_gradient
df_to_excel_with_gradient(df_all_user.head(5), 'acc_by_tile_group_best_user.xlsx')
df_to_excel_with_gradient(df_all_prod.head(5), 'acc_by_tile_group_best_producer.xlsx')
df_to_excel_with_gradient(df_all_no_lc, 'acc_by_tile_group_best_no_lc.xlsx')
df_to_excel_with_gradient(df_all_prod[df_all_prod['hls_type'] == 'veg'].head(5), 'acc_by_tile_group_best_producer_veg.xlsx')


In [28]:
df_acc_expanded.head()

,umd_name,acc,lc_class,hls_type,acc_type
33,gen__47TNH,1.0,surface_water,gen,producer_acc
23,fire__22LDR,1.0,wetland_vegetation_sparse,gen,total_acc
60,treelossTF__35VLF,1.0,desert,veg,total_acc
59,treelossTF__33NUF,1.0,wetland_vegetation_sparse,gen,total_acc
33,gen__47TNH,1.0,dense_short_vegetation,gen,total_acc


In [30]:
n_to_show = 25

for a in ['user_acc', 'producer_acc', 'f1']:
    print('########')
    print(a)
    df_acc_temp = df_acc_expanded[df_acc_expanded['acc_type'] == a].reset_index(drop=True)
    df_acc_temp = df_acc_temp[df_acc_temp['acc'] < 1]
    df_acc_temp = df_acc_temp[df_acc_temp['lc_class'] == 'all_classes']
    df_acc_temp = df_acc_temp.sort_values(by='acc', ascending=False).reset_index(drop=True)
    df_acc_temp = df_acc_temp[df_acc_temp['hls_type'] == 'veg'].reset_index(drop=True)
    print('best')
    print(df_acc_temp.head(n_to_show))
    print('worst')
    print(df_acc_temp.tail(n_to_show))

########
user_acc
best
                umd_name       acc     lc_class hls_type  acc_type
0            fire__20LNJ  0.901472  all_classes      veg  user_acc
1     treelosswet__50NPL  0.797787  all_classes      veg  user_acc
2   oldcrop_short__51KVV  0.736749  all_classes      veg  user_acc
3     treelosswet__20LMM  0.712562  all_classes      veg  user_acc
4   builtnewalert__22KFA  0.690530  all_classes      veg  user_acc
5   builtnewalert__22MGB  0.668276  all_classes      veg  user_acc
6     treelosswet__20KPF  0.647369  all_classes      veg  user_acc
7            fire__22LDR  0.616397  all_classes      veg  user_acc
8     treelosswet__20KNG  0.607144  all_classes      veg  user_acc
9     treelosswet__17SQT  0.593400  all_classes      veg  user_acc
10    treelosswet__48MUD  0.584045  all_classes      veg  user_acc
11           fire__52UCF  0.582900  all_classes      veg  user_acc
12  builtnewalert__22LDQ  0.575186  all_classes      veg  user_acc
13       wetshort__22LEQ  0.573601  all